# Alpha Defect Detection

**Goal:** Identify steel coils with alpha defects in a hot-rolling process.

**Dataset:** 1352 train / 339 test samples, 49 sensor features, 4.88% defect rate (66 defects).

## Approach
1. **Feature engineering** — temperature progressions, pairwise interactions, percentile ranks
2. **Anomaly detection features** — IsolationForest + LOF scores (unsupervised, no label leakage)
3. **Optuna tuning** — 60-trial hyperparameter search on honest OOF Average Precision
4. **4-model rank ensemble** — XGBoost + LightGBM + RandomForest + ExtraTrees
5. **Top-K selection** — flag the top-17 ranked test coils (avoids threshold calibration issues)
6. **SMOTE inside each CV fold** — no data leakage

**Top features by univariate AUC:** X13 (0.83), X10 (0.82), X32 (0.81), X30 (0.81), X36 (0.81), X31 (0.80)

In [ ]:
# -*- coding: utf-8 -*-
import os, sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score,
                             confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              IsolationForest)
from sklearn.neighbors import LocalOutlierFactor
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM not found - XGB will be used in its place')

print('Libraries loaded.')

In [ ]:
# 1. LOAD DATA
train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')

feature_cols = [c for c in train.columns if c.startswith('X')]
y          = train['Y'].astype(int).values
test_ids   = test['CoilID'].values
n_pos      = y.sum()
n_neg      = (y == 0).sum()
expected   = round(len(test) * y.mean())

print(f'Train {train.shape}: {n_pos} defects ({100*y.mean():.2f}%)')
print(f'Test  {test.shape}: expected ~{expected} defects')
print(f'Imbalance ratio: {n_neg/n_pos:.1f}:1')

# Imputation
imp      = SimpleImputer(strategy='median')
X_raw    = imp.fit_transform(train[feature_cols])
X_te_raw = imp.transform(test[feature_cols])

In [ ]:
# 2. FEATURE ENGINEERING
def engineer(X, fcols):
    f = {c: X[:, i] for i, c in enumerate(fcols)}
    e = {}

    # Temperature progression X1-X9
    tmp = np.array([f[f'X{i}'] for i in range(1, 10)])
    e['temp_mean']  = tmp.mean(0)
    e['temp_std']   = tmp.std(0)
    e['temp_range'] = tmp.max(0) - tmp.min(0)
    e['temp_drop']  = f['X1'] - f['X9']
    e['temp_cv']    = e['temp_std'] / (e['temp_mean'] + 1e-9)
    for i in range(1, 9):
        e[f'tdrop_{i}'] = f[f'X{i}'] - f[f'X{i+1}']

    # Key features — log and squared transforms (top AUC features)
    for col in ['X13', 'X10', 'X32', 'X30', 'X36', 'X31', 'X34']:
        e[f'{col}_log'] = np.log1p(np.clip(f[col], 0, None))
        e[f'{col}_sq']  = f[col] ** 2

    # Pairwise interactions between top-5 features
    top = ['X13', 'X10', 'X32', 'X30', 'X36']
    for i in range(len(top)):
        for j in range(i+1, len(top)):
            a, b = top[i], top[j]
            e[f'{a}x{b}'] = f[a] * f[b]
            e[f'{a}d{b}'] = f[a] - f[b]

    # Rolling reduction X23-X33
    red = np.array([f[f'X{i}'] for i in range(23, 34)])
    e['red_mean']  = red.mean(0)
    e['red_std']   = red.std(0)
    e['red_range'] = red.max(0) - red.min(0)
    e['red_cv']    = e['red_std'] / (e['red_mean'] + 1e-9)

    # Force X34-X36
    e['force_sum']   = f['X34'] + f['X35'] + f['X36']
    force_arr = np.array([f['X34'], f['X35'], f['X36']])
    e['force_range'] = force_arr.max(0) - force_arr.min(0)

    # Speed X17-X22
    spd = np.array([f[f'X{i}'] for i in range(17, 23)])
    e['speed_mean']  = spd.mean(0)
    e['speed_std']   = spd.std(0)
    e['speed_range'] = spd.max(0) - spd.min(0)

    # Vibration / quality sensors X41-X49
    qual_keys = [f'X{i}' for i in range(41, 50) if f'X{i}' in f]
    qual = np.array([f[k] for k in qual_keys])
    e['qual_mean']  = qual.mean(0)
    e['qual_std']   = qual.std(0)
    e['qual_range'] = qual.max(0) - qual.min(0)

    # Cross-group interactions
    e['temp_x_force'] = e['temp_mean'] * e['force_sum']
    e['speed_x_red']  = e['speed_mean'] * e['red_mean']
    e['X13_x_qual']   = f['X13'] * e['qual_mean']
    e['X36_x_red']    = f['X36'] * e['red_mean']

    # Global statistics
    e['g_mean']  = X.mean(1)
    e['g_std']   = X.std(1)
    e['g_range'] = X.max(1) - X.min(1)
    e['g_skew']  = pd.DataFrame(X).skew(axis=1).values

    # Percentile rank (scale-free position in distribution)
    for col in ['X13', 'X10', 'X32', 'X30', 'X36', 'X31']:
        v = f[col]
        e[f'{col}_pct'] = np.argsort(np.argsort(v)) / max(len(v) - 1, 1)

    return np.hstack([X, np.column_stack(list(e.values()))])

X_eng    = engineer(X_raw,    feature_cols)
X_te_eng = engineer(X_te_raw, feature_cols)
print(f'Features: {len(feature_cols)} raw -> {X_eng.shape[1]} engineered')

In [ ]:
# 3. ANOMALY DETECTION FEATURES (unsupervised — no label leakage)
print('Computing anomaly detection scores...')
sc_base  = RobustScaler()
X_sc     = sc_base.fit_transform(X_eng)
X_te_sc  = sc_base.transform(X_te_eng)

iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_sc)
iso_tr = (-iso.score_samples(X_sc)).reshape(-1, 1)
iso_te = (-iso.score_samples(X_te_sc)).reshape(-1, 1)

lof = LocalOutlierFactor(n_neighbors=15, contamination=0.05, novelty=True)
lof.fit(X_sc)
lof_tr = (-lof.score_samples(X_sc)).reshape(-1, 1)
lof_te = (-lof.score_samples(X_te_sc)).reshape(-1, 1)

X    = np.hstack([X_eng, iso_tr, lof_tr])
X_te = np.hstack([X_te_eng, iso_te, lof_te])
print(f'Augmented features: {X.shape[1]}')

In [ ]:
# 4. OPTUNA HYPERPARAMETER SEARCH
#    Objective: maximise OOF Average Precision (no data leakage)
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def oof_ap(params, X, y):
    skf_ = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof  = np.zeros(len(y))
    for tr_idx, val_idx in skf_.split(X, y):
        Xtr, Xval = X[tr_idx], X[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]
        sm = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=42)
        Xtr_r, ytr_r = sm.fit_resample(Xtr, ytr)
        sc = RobustScaler()
        m  = XGBClassifier(**params, eval_metric='aucpr', n_jobs=-1)
        m.fit(sc.fit_transform(Xtr_r), ytr_r)
        oof[val_idx] = m.predict_proba(sc.transform(Xval))[:, 1]
    return average_precision_score(y, oof)

def objective(trial):
    return oof_ap({
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 6),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5.0, 30.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 3.0),
        'random_state': 42,
    }, X, y)

print('Running Optuna (60 trials)...')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=False)

best_params = {**study.best_params, 'eval_metric': 'aucpr', 'n_jobs': -1, 'random_state': 42}
print(f'Best OOF AP: {study.best_value:.4f}')
print(f'Best params: {best_params}')

In [ ]:
# 5. FULL ENSEMBLE CV  (XGBoost + LightGBM + RandomForest + ExtraTrees)
oof = {m: np.zeros(len(y)) for m in ['xgb', 'lgbm', 'rf', 'et']}
te  = {m: [] for m in ['xgb', 'lgbm', 'rf', 'et']}

lgbm_p = dict(
    n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'], num_leaves=2**best_params['max_depth']-1,
    subsample=best_params['subsample'], colsample_bytree=best_params['colsample_bytree'],
    min_child_samples=max(1, best_params['min_child_weight']),
    scale_pos_weight=best_params['scale_pos_weight'],
    reg_alpha=best_params['reg_alpha'], reg_lambda=best_params['reg_lambda'],
    n_jobs=-1, random_state=42, verbose=-1
) if HAS_LGBM else None

rf_p = dict(n_estimators=500, max_depth=10, min_samples_leaf=1,
            max_features='sqrt', class_weight='balanced_subsample',
            n_jobs=-1, random_state=42)
et_p = dict(n_estimators=500, max_depth=10, min_samples_leaf=1,
            max_features='sqrt', class_weight='balanced_subsample',
            n_jobs=-1, random_state=42)

print(f'Training {N_FOLDS}-fold ensemble...')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    Xtr, Xval = X[tr_idx], X[val_idx]
    ytr, yval = y[tr_idx], y[val_idx]

    sm = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=42)
    Xtr_r, ytr_r = sm.fit_resample(Xtr, ytr)

    sc = RobustScaler()
    Xtr_s, Xval_s, Xte_s = sc.fit_transform(Xtr_r), sc.transform(Xval), sc.transform(X_te)

    m_xgb = XGBClassifier(**best_params)
    m_xgb.fit(Xtr_s, ytr_r)
    oof['xgb'][val_idx] = m_xgb.predict_proba(Xval_s)[:, 1]
    te['xgb'].append(m_xgb.predict_proba(Xte_s)[:, 1])

    if HAS_LGBM:
        m_lgbm = LGBMClassifier(**lgbm_p)
        m_lgbm.fit(Xtr_s, ytr_r)
        oof['lgbm'][val_idx] = m_lgbm.predict_proba(Xval_s)[:, 1]
        te['lgbm'].append(m_lgbm.predict_proba(Xte_s)[:, 1])
    else:
        oof['lgbm'][val_idx] = oof['xgb'][val_idx]
        te['lgbm'].append(te['xgb'][-1])

    m_rf = RandomForestClassifier(**rf_p)
    m_rf.fit(Xtr_s, ytr_r)
    oof['rf'][val_idx] = m_rf.predict_proba(Xval_s)[:, 1]
    te['rf'].append(m_rf.predict_proba(Xte_s)[:, 1])

    m_et = ExtraTreesClassifier(**et_p)
    m_et.fit(Xtr_s, ytr_r)
    oof['et'][val_idx] = m_et.predict_proba(Xval_s)[:, 1]
    te['et'].append(m_et.predict_proba(Xte_s)[:, 1])

    print(f"  Fold {fold+1}/{N_FOLDS} | XGB AUC={roc_auc_score(yval, oof['xgb'][val_idx]):.4f}"
          f" AP={average_precision_score(yval, oof['xgb'][val_idx]):.4f} | defects={yval.sum()}")

In [ ]:
# 6. RANK-BASED BLEND
#    Rank normalisation makes blending more robust than raw probability averaging
def rank_norm(arr):
    r = np.argsort(np.argsort(arr)).astype(float)
    return r / (len(r) - 1)

W = dict(xgb=0.35, lgbm=0.35, rf=0.15, et=0.15)
oof_blend = sum(W[m] * rank_norm(oof[m]) for m in W)
te_avg    = {m: np.mean(te[m], axis=0) for m in te}
te_blend  = sum(W[m] * rank_norm(te_avg[m]) for m in W)

print('--- OOF Performance (no data leakage) ---')
for m in ['xgb', 'lgbm', 'rf', 'et']:
    a = roc_auc_score(y, oof[m]);  p = average_precision_score(y, oof[m])
    print(f'  {m:5s}: AUC={a:.4f}  AP={p:.4f}')
ab = roc_auc_score(y, oof_blend); pb = average_precision_score(y, oof_blend)
print(f'  blend: AUC={ab:.4f}  AP={pb:.4f}')

In [ ]:
# 7. OOF THRESHOLD ANALYSIS
print(f"{'Thresh':7} {'Recall':8} {'Prec':8} {'F1':8} {'FN':5} {'FP':5} {'Test+':7}")
print('-' * 55)
best_t = 0.3; best_f1 = 0
for t in np.arange(0.05, 0.95, 0.01):
    p    = (oof_blend >= t).astype(int)
    rec  = recall_score(y, p, zero_division=0)
    prec = precision_score(y, p, zero_division=0)
    f1   = f1_score(y, p, zero_division=0)
    cm   = confusion_matrix(y, p)
    fn, fp = cm[1][0], cm[0][1]
    te_n = (te_blend >= t).sum()
    if round(t, 2) in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
        print(f'{t:7.2f} {rec:8.4f} {prec:8.4f} {f1:8.4f} {fn:5d} {fp:5d} {te_n:7d}')
    if f1 > best_f1:
        best_f1 = f1; best_t = t

p_ref = (oof_blend >= best_t).astype(int)
print(f'\nBest F1 threshold: {best_t:.2f}')
print(f'OOF Recall={recall_score(y, p_ref):.4f}  Prec={precision_score(y, p_ref, zero_division=0):.4f}')

In [ ]:
# 8. FINAL PREDICTION: TOP-K SELECTION
#    Flag the top-K highest-ranked test coils.
#    Avoids threshold calibration issues common with imbalanced datasets.
K = min(max(expected, 15), 30)   # 15-30 coils
top_k_idx   = np.argsort(te_blend)[-K:]
final_preds = np.zeros(len(te_blend), dtype=int)
final_preds[top_k_idx] = 1

print(f'OOF blend AUC={ab:.4f}, AP={pb:.4f}')
print(f'Top-K selection: K={K} (expected ~{expected})')
print(f'Test defects flagged: {final_preds.sum()}')

# 9. SAVE SUBMISSION
sub = pd.DataFrame({'CoilID': test_ids, 'Y': final_preds})
assert sub.shape == (339, 2)
assert list(sub.columns) == ['CoilID', 'Y']
sub.to_csv('submission.csv', index=False)

print()
print('=' * 65)
print('SAVED: submission.csv')
print('=' * 65)
print(f"Defect=1: {final_preds.sum()}  |  No-defect=0: {(final_preds==0).sum()}")
print(f"Flagged CoilIDs: {sub[sub['Y']==1]['CoilID'].tolist()}")
print('\n[DONE] Ready for upload!')